# Notebook 10 — Machine Learning Dataset Creation

This notebook prepares the final feature matrices (X) and target vectors (y), applies categorical one-hot encoding to districts (estates), and splits the dataset chronologically (train on 2019–2023, test on 2024–2025) to prevent leakage.

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Processed Folder:", config.PROCESSED_DIR)

Processed Folder: D:\A1\coco\Coconut_Yield_Prediction\processed


## 2. Load Features Dataset

In [2]:
df = pd.read_csv(config.PROCESSED_DIR / "master_features.csv")
print("Features dataset shape:", df.shape)

Features dataset shape: (608, 37)


## 3. Define Features (X) and Target (y)

In [3]:
# Drop unneeded index/target columns
drop_cols = ["Avg_Nuts_Palm", "Palms_Count"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[["District", "Year", "Month", "Avg_Nuts_Palm"]].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (608, 35)
y shape: (608, 4)


## 4. One-Hot Encoding

In [4]:
cat_cols = ["District"]
if "zone" in X.columns:
    cat_cols.append("zone")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Ensure dummy columns are integers (0/1)
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

print("Shape after One-Hot Encoding:", X_encoded.shape)
print(X_encoded.head())

Shape after One-Hot Encoding: (608, 41)
   Year  Month  Rainfall  Temperature  Humidity  Soil_Moisture_Top  \
0  2019      1      6.82        25.29     79.06               0.60   
1  2019      2     79.24        27.23     76.37               0.50   
2  2019      3     24.18        29.14     64.64               0.40   
3  2019      4    133.50        29.68     69.60               0.42   
4  2019      5     30.38        28.97     78.03               0.47   

   Soil_Moisture_Root  Soil_Moisture_Diff  Solar_Radiation  Temperature_Max  \
0                0.60                0.00            19.71            31.54   
1                0.52                0.02            20.34            34.40   
2                0.45                0.05            24.14            36.46   
3                0.45                0.03            21.75            36.51   
4                0.49                0.02            21.86            33.29   

   ...  Rain_SoilMoist_Interaction  Temp_Humidity_Interaction  \

## 5. Chronological Train-Test Split

To validate agricultural forecasting models realistically:
- **Train Set**: Years 2019–2023
- **Test Set**: Years 2024–2025

In [5]:
train_mask = X_encoded["Year"] <= 2023
test_mask = X_encoded["Year"] > 2023

X_train = X_encoded[train_mask].copy()
X_test = X_encoded[test_mask].copy()

y_train = y[train_mask]["Avg_Nuts_Palm"].copy()
y_test = y[test_mask]["Avg_Nuts_Palm"].copy()

# Drop temporal index columns from model inputs
X_train = X_train.drop(columns=["Year", "Month"])
X_test = X_test.drop(columns=["Year", "Month"])

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}   | y_test shape: {y_test.shape}")

X_train shape: (480, 39) | y_train shape: (480,)
X_test shape: (128, 39)   | y_test shape: (128,)


## 6. Export Datasets

In [6]:
X_train.to_csv(config.PROCESSED_DIR / "X_train.csv", index=False)
X_test.to_csv(config.PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(config.PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(config.PROCESSED_DIR / "y_test.csv", index=False)

print("[SUCCESS] Train and test matrices exported successfully.")

[SUCCESS] Train and test matrices exported successfully.
